# Model Training (reference data are videos)

In [1]:
# filename = 'ADTC_8_adz_raw_20260309'
# datum = filename.split('_')[-1]
# yy = int(datum[0:4])
# mm = int(datum[4:6])
# dd = int(datum[6:8])

# filename = 'D_05_adz_raw_20260301'
# datum = filename.split('_')[-1]
# yy = int(datum[0:4])
# mm = int(datum[4:6])
# dd = int(datum[6:8])

filename = 'ADTC1_adz_raw_20260615'

In [ ]:
from FTP.video_download import video_downloader

datum = video_downloader(filename)

Connection with geo-amberg.ch
Successful access to /cam/2026/06/15
ADTC Rastatt_00_20260615064512.mp4 successfully downloaded
ADTC Rastatt_00_20260615141717.mp4 successfully downloaded
ADTC Rastatt_00_20260615134915.mp4 successfully downloaded
ADTC Rastatt_00_20260615064704.mp4 successfully downloaded
ADTC Rastatt_00_20260615071932.mp4 successfully downloaded
ADTC Rastatt_00_20260615222518.mp4 successfully downloaded
ADTC Rastatt_00_20260615145359.mp4 successfully downloaded
ADTC Rastatt_00_20260615094443.mp4 successfully downloaded
ADTC Rastatt_00_20260615160133.mp4 successfully downloaded
ADTC Rastatt_00_20260615081709.mp4 successfully downloaded
ADTC Rastatt_00_20260615162548.mp4 successfully downloaded
ADTC Rastatt_00_20260615212052.mp4 successfully downloaded
ADTC Rastatt_00_20260615062926.mp4 successfully downloaded
ADTC Rastatt_00_20260615181659.mp4 successfully downloaded
ADTC Rastatt_00_20260615194650.mp4 successfully downloaded
ADTC Rastatt_00_20260615223227.mp4 successfully 

## DataFrame for each video (path and timestamp)

In [3]:
import os
import pandas as pd
from pathlib import Path

# change to the file directory
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

src = root / f'VIDEO/{datum}'

rows = []

for f in os.listdir(src):
    if f.endswith('.mp4'):
        file = os.path.join(src, f)
        t = f[-18:-1]
        year = t[0:4]
        month = t[4:6]
        day = t[6:8]
        hour = t[8:10]
        minute = t[10:12]
        seconde = t[12:14]

        timestamp_video = f'{year}-{month}-{day} {hour}:{minute}:{seconde}'
        
        rows.append({
            "video": file,
            "timestamp_video": pd.to_datetime(timestamp_video)
        })

df_video = pd.DataFrame(rows)
df_video.head()

,video,timestamp_video
0,c:\git\RailwAI\VIDEO\20260615\ADTC Rastatt_00_...,2026-06-15 00:27:42
1,c:\git\RailwAI\VIDEO\20260615\ADTC Rastatt_00_...,2026-06-15 00:32:56
2,c:\git\RailwAI\VIDEO\20260615\ADTC Rastatt_00_...,2026-06-15 00:40:00
3,c:\git\RailwAI\VIDEO\20260615\ADTC Rastatt_00_...,2026-06-15 00:40:17
4,c:\git\RailwAI\VIDEO\20260615\ADTC Rastatt_00_...,2026-06-15 00:54:17


In [4]:
# df_train_data = pd.read_csv(f'DataFrame/df_merged_{filename}.csv')

# df_video['timestamp_video'] = pd.to_datetime(df_video['timestamp_video'])
# df_train_data['timestamp'] = pd.to_datetime(df_train_data['timestamp'])

# df_video_merged = pd.merge_asof(
#     df_video.sort_values('timestamp_video'),
#     df_train_data.sort_values('timestamp'),
#     left_on='timestamp_video',
#     right_on='timestamp',
#     direction='nearest',
#     tolerance=pd.Timedelta('10s')
# )

# df_video_merged = df_video_merged.dropna(subset=['file'])

# cols = [c for c in df_video_merged.columns if c not in ["video", "timestamp_video"]] + ["video", "timestamp_video"]
# df_video_merged = df_video_merged[cols]

# df_video_merged.head(5)

In [5]:
import pandas as pd

# Load data
df_train_data = pd.read_csv(f'DataFrame/df_merged_{filename}.csv')

# Convert timestamps
df_video['timestamp_video'] = pd.to_datetime(df_video['timestamp_video'])
df_train_data['timestamp'] = pd.to_datetime(df_train_data['timestamp'])

# Merge each video with the nearest train within 10 seconds
df_video_merged = pd.merge_asof(
    df_video.sort_values('timestamp_video'),
    df_train_data.sort_values('timestamp'),
    left_on='timestamp_video',
    right_on='timestamp',
    direction='nearest',
    tolerance=pd.Timedelta('10s')
)

# Remove videos with no matching train
df_video_merged = df_video_merged.dropna(subset=['file'])

# Find files with multiple videos
duplicate_counts = df_video_merged['file'].value_counts()
duplicated_files = duplicate_counts[duplicate_counts > 1]

print(f"{len(duplicated_files)} file(s) have more than one matching video.")

# Print timestamps of duplicated videos
if len(duplicated_files) > 0:
    print("\nFiles with multiple videos:")
    
    for file_name in duplicated_files.index:
        print(f"\nFile: {file_name}")
        
        videos = df_video_merged[df_video_merged['file'] == file_name]
        
        for _, row in videos.sort_values('timestamp_video').iterrows():
            print(f"  Video: {row['video']}")
            print(f"  Timestamp: {row['timestamp_video']}")

# Keep only the first video for each file
df_video_merged = (
    df_video_merged
    .sort_values('timestamp_video')
    .drop_duplicates(subset='file', keep='first')
)

# Reorder columns
cols = [c for c in df_video_merged.columns if c not in ["video", "timestamp_video"]] + ["video", "timestamp_video"]
df_video_merged = df_video_merged[cols]

# Preview
print("\nFinal dataframe:")
print(df_video_merged.head())

print(f"\nFinal number of unique files: {df_video_merged['file'].nunique()}")
print(f"Final number of rows: {len(df_video_merged)}")

5 file(s) have more than one matching video.

Files with multiple videos:

File: ADTC1_adz_raw_20260615_0614
  Video: c:\git\RailwAI\VIDEO\20260615\ADTC Rastatt_00_20260615061255.mp4
  Timestamp: 2026-06-15 06:12:55
  Video: c:\git\RailwAI\VIDEO\20260615\ADTC Rastatt_00_20260615061304.mp4
  Timestamp: 2026-06-15 06:13:04

File: ADTC1_adz_raw_20260615_1004
  Video: c:\git\RailwAI\VIDEO\20260615\ADTC Rastatt_00_20260615100254.mp4
  Timestamp: 2026-06-15 10:02:54
  Video: c:\git\RailwAI\VIDEO\20260615\ADTC Rastatt_00_20260615100259.mp4
  Timestamp: 2026-06-15 10:02:59

File: ADTC1_adz_raw_20260615_0926
  Video: c:\git\RailwAI\VIDEO\20260615\ADTC Rastatt_00_20260615092500.mp4
  Timestamp: 2026-06-15 09:25:00
  Video: c:\git\RailwAI\VIDEO\20260615\ADTC Rastatt_00_20260615092503.mp4
  Timestamp: 2026-06-15 09:25:03

File: ADTC1_adz_raw_20260615_2117
  Video: c:\git\RailwAI\VIDEO\20260615\ADTC Rastatt_00_20260615211606.mp4
  Timestamp: 2026-06-15 21:16:06
  Video: c:\git\RailwAI\VIDEO\2026061

In [6]:
import csv
import os

# change to the file directory
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

src = root / f'VIDEO/{datum}/'

rest = []

for f in os.listdir(src):
    if f == 'reference_video.csv':
        continue
    file = os.path.join(src, f)

    if file not in df_video_merged['video'].values:
        os.remove(file)
    else:
        rest.append(file)
    
output_csv = os.path.join(src, 'reference_video.csv')

if os.path.exists(output_csv):
    print('reference_video.csv already exists')
    
else:
    with open(output_csv, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile, delimiter=';')  # use ; as separator
        writer.writerow(['video', 'train_type'])
        for vid in rest:
            writer.writerow([vid, ''])

reference_video.csv already exists


# Manually assign the train type in the CSV file

In [7]:
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

csv_train = root / f'VIDEO/{datum}/reference_video.csv'
df_ref = pd.read_csv(csv_train, sep=';')

df_ref = df_ref.drop(
    columns=[col for col in df_ref.columns if col in df_video_merged.columns and col != 'video']
)

df_video_merged = df_video_merged.merge(
    df_ref,
    on='video',
    how='left'
)

output_dir = root / "DataframeVideo"
output_dir.mkdir(exist_ok=True)

df_video_merged.to_csv(
    output_dir / f"df_video_merged_{datum}.csv",
    index=False
)

df_video_merged.head(5)

,file,timestamp_raw,timestamp,timestamp_serie,delta_t,velocity,length,total_length,first_peak_raw,video,timestamp_video,train_type
0,ADTC1_adz_raw_20260615_0028,2026-06-15 00:27:32.099,2026-06-15 00:27:42.139,"[Timestamp('2026-06-15 00:27:42.139000'), Time...","[0.12, 0.76, 0.12, 0.21, 0.12, 0.785, 0.117, 0...","[21.374729691864434, 21.097272270334212, 21.23...","[2.564967563023732, 16.033926925454, 2.5476062...",99.078036,9.922365,c:\git\RailwAI\VIDEO\20260615\ADTC Rastatt_00_...,2026-06-15 00:27:42,Trieb
1,ADTC1_adz_raw_20260615_0034,2026-06-15 00:32:46.234,2026-06-15 00:32:56.261,"[Timestamp('2026-06-15 00:32:56.261000'), Time...","[0.121, 0.37, 0.122, 0.221, 0.084, 0.344, 0.08...","[20.83643307047563, 21.136194492681717, 20.978...","[2.521208401527551, 7.820391962292235, 2.55937...",2925.141415,9.917025,c:\git\RailwAI\VIDEO\20260615\ADTC Rastatt_00_...,2026-06-15 00:32:56,Lok
2,ADTC1_adz_raw_20260615_0041,2026-06-15 00:39:50.668,2026-06-15 00:40:00.715,"[Timestamp('2026-06-15 00:40:00.715000'), Time...","[0.124, 0.37, 0.124]","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan]",NaN,9.936346,c:\git\RailwAI\VIDEO\20260615\ADTC Rastatt_00_...,2026-06-15 00:40:00,Lok
3,ADTC1_adz_raw_20260615_0055,2026-06-15 00:54:06.714,2026-06-15 00:54:16.755,"[Timestamp('2026-06-15 00:54:16.755000'), Time...","[0.105, 0.583, 0.12, 0.616, 0.12, 0.583, 0.108]","[22.261426993831982, 22.226805702037897, 21.67...","[2.337449834352358, 12.958227724288093, 2.6006...",49.536474,9.931430,c:\git\RailwAI\VIDEO\20260615\ADTC Rastatt_00_...,2026-06-15 00:54:17,Trieb
4,ADTC1_adz_raw_20260615_0102,2026-06-15 01:01:32.921,2026-06-15 01:01:42.948,"[Timestamp('2026-06-15 01:01:42.948000'), Time...","[0.146, 0.328, 0.144, 0.305, 0.122, 0.378, 0.118]","[21.088180814638804, 20.90724385451653, 21.513...","[3.0788743989372653, 6.857575984281422, 3.0980...",32.291124,9.912503,c:\git\RailwAI\VIDEO\20260615\ADTC Rastatt_00_...,2026-06-15 01:01:42,Lok


## Inference New Data

In [17]:
from dataset import InferenceImageDataset

inference_dataset = InferenceImageDataset(
    img_dir="C:/git/RailwAI/images/inference",
    transform=transform
)

inference_dataloader = DataLoader(
    inference_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

print("Inference images:", len(inference_dataset))

Inference images: 75


Load trained model

In [18]:
model.load_state_dict(
    torch.load("efficientnet_b3_traintyp.pth", map_location=device)
)

model.to(device)
model.eval()

classes = list(train_dataset.class_to_idx.keys())

Run inference

In [19]:
import pandas as pd
import torch

results = []

with torch.no_grad():
    for imgs, paths in inference_dataloader:

        imgs = imgs.to(device)

        outputs = model(imgs)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        for path, pred, prob in zip(paths, preds, probs):

            pred_class = classes[pred.item()]
            confidence = prob[pred.item()].item() * 100

            print(f"{path} -> {pred_class}: {confidence:.2f}%")

            results.append({
                "image": path,
                "prediction": pred_class,
                "confidence": confidence
            })


df_results = pd.DataFrame(results)

df_results.to_csv(
    "inference_results.csv",
    index=False
)

df_results.head()

C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0035.png -> Lok: 99.21%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0113.png -> Trieb: 100.00%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0133.png -> Trieb: 69.56%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0151.png -> Trieb: 99.99%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0230.png -> Trieb: 98.44%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0332.png -> Trieb: 68.89%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0351.png -> Trieb: 99.85%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0431.png -> Trieb: 90.92%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0536.png -> Lok: 98.77%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0606.png -> Lok: 98.61%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0635.png -> Lok: 94.66%
C:/git/RailwAI/images/inference\D_05_adz_raw_20260301_0735.png -> Lok: 97.35%
C:/git/RailwAI/images/inference\D_05_adz_raw_2026

,image,prediction,confidence
0,C:/git/RailwAI/images/inference\D_05_adz_raw_2...,Lok,99.214923
1,C:/git/RailwAI/images/inference\D_05_adz_raw_2...,Trieb,100.000000
2,C:/git/RailwAI/images/inference\D_05_adz_raw_2...,Trieb,69.556230
3,C:/git/RailwAI/images/inference\D_05_adz_raw_2...,Trieb,99.991691
4,C:/git/RailwAI/images/inference\D_05_adz_raw_2...,Trieb,98.441327
